# Load AI-READI clinical tables

Set the dataset path below, then run the cells in order. This notebook reads your local files. It shows clinical mapping, a participant-by-feature table and CSV export. Saved outputs are cleared in this copy.

In [ ]:
from pathlib import Path
import pandas as pd
from aireadi_preprocessing import load_clinical
from aireadi_preprocessing.tables import read_table, measurement_mapping

dataset = Path("/path/to/ai-readi")
output = Path("/path/to/local-tables")

## Inspect the clinical codes

The [AI-READI mapping JSON](https://github.com/AI-READI/ai-readi-docs/blob/main/docs/static/json/mappings.json) defines the identifiers. This is an excerpt of its HbA1c entry:

```json
{
  "FIELD_ID": "import_hba1c",
  "SRC_CODE": "4548-4",
  "TARGET_CONCEPT_ID": "3004410",
  "TARGET_CONCEPT_NAME": "Hemoglobin A1c/Hemoglobin.total in Blood",
  "TARGET_DOMAIN_ID": "Measurement"
}
```

- `FIELD_ID` identifies the original collection field.
- `SRC_CODE` is the source terminology code, here the LOINC code for HbA1c.
- `TARGET_CONCEPT_ID` is the OMOP identifier matched to `measurement_concept_id` in the data. For this measurement, use `3004410`, not `4548-4`.
- `TARGET_CONCEPT_NAME` gives the clinical meaning. The [lab dictionary](https://github.com/AI-READI/ai-readi-docs/blob/main/docs/static/json/clinicalLabData.json) also gives the shorter `Name` (HbA1c) and `Units` (%).

The package extracts the working mapping from your downloaded `measurement.csv`:

| Input field | Extracted mapping field | Role |
| --- | --- | --- |
| `measurement_concept_id` | `concept_id` | Identifier used to match rows |
| `measurement_source_value` | `name` | Output column label, such as `HbA1c (%)` |
| `unit_source_value` | `unit` | Unit checked before conversion |

1. Select rows with a numeric value and extract the distinct code/label/unit combinations.
2. Save that dictionary as `clinical_mapping.csv`. The documentation JSON explains the codes; it is not downloaded or parsed during conversion.
3. Match measurement rows to the dictionary by code, then pivot `value_as_number` into columns using `person_id` as the row key.

For example, these fictional input rows:

| person_id | measurement_concept_id | measurement_source_value | unit_source_value | value_as_number |
| --- | --- | --- | --- | --- |
| 0007 | 3004410 | HbA1c (%) | % | 5.5 |
| 0012 | 3004410 | HbA1c (%) | % | 6.0 |

produce:

| person_id | HbA1c (%) |
| --- | --- |
| 0007 | 5.5 |
| 0012 | 6.0 |

The next cell extracts the mapping from your local measurement file.


In [ ]:
measurements = read_table(dataset / "clinical_data" / "measurement.csv")
mapping = measurement_mapping(measurements)
mapping.head()

## Load the clinical table

The package matches measurement rows by code and pivots their numeric values into columns. Participant IDs remain strings, so leading zeros are preserved. A repeated participant/code pair stops the pivot; choose a visit or an aggregation before continuing.

In [ ]:
clinical = load_clinical(dataset)
clinical.head()

## Choose measurements and column names

This selection uses HbA1c, code `3004410`, with unit `%`. Change the rows to choose other measurements after checking the mapping. An absent measurement produces an empty column; a unit mismatch raises an error.

In [ ]:
selected_mapping = pd.DataFrame([
    {"concept_id": "3004410", "name": "HbA1c (%)", "unit": "%"},
])
selected = load_clinical(dataset, mapping=selected_mapping)
selected.head()

## Save and reload a CSV

Choose a local output directory outside the source dataset and repository. Keep `person_id` as text when reading the result back.

In [ ]:
output.mkdir(parents=True, exist_ok=True)
selected.to_csv(output / "clinical.csv", index=False)
selected_mapping.to_csv(output / "clinical_mapping.csv", index=False)
reloaded = pd.read_csv(output / "clinical.csv", dtype={"person_id": str})
reloaded.head()